# Módulo métricas de congestión
Fórmulas estándar (CEJA / Banco Mundial):

| indicador | fórmula |
|---|---|
| tasa de resolución (clearance rate) | resueltas / ingresadas |
| tasa de congestión | atendidas / resueltas |
| tasa de pendencia | pendientes_fin / atendidas |
| duración estimada (días) | pendientes_fin / resueltas × 365 |

`ingresadas` es la **suma de todas las formas de ingreso**, no solo `nuevas_ingresadas`.
Se verifica con la identidad del anuario: ingresadas = atendidas − pendientes_inicio.

In [ ]:
import numpy as np
import pandas as pd

COLUMNAS_INGRESOS = [
    "readecuadas_ley_439",
    "nuevas_ingresadas",
    "recibidas_excusa_recusacion",
    "preliminares_formalizados",
    "cautelares_formalizados",
    "ingresadas_conversion_acciones",
    "ingresadas_reenvio",
    "otras_formas_ingreso",
]


def calcular_ingresos_totales(df):
    presentes = []
    for c in COLUMNAS_INGRESOS:
        if c in df.columns:
            presentes.append(c)
    return df[presentes].fillna(0).sum(axis=1)


def verificar_identidad_ingresos(df, tolerancia=0.0, mascara=None):
    # Devuelve solo las filas donde la suma de ingresos no cierra con atendidas - pendientes_inicio.
    suma = calcular_ingresos_totales(df)
    comparables = df["atendidas"].notna() & df["pendientes_inicio"].notna()
    if mascara is not None:
        comparables = comparables & mascara
    identidad = df["atendidas"].astype(float) - df["pendientes_inicio"].astype(float)
    diferencia = (identidad - suma).abs()
    descuadre = comparables & (diferencia > tolerancia)
    salida = df.loc[descuadre].copy()
    salida["ingresos_por_suma"] = suma[descuadre]
    salida["ingresos_por_identidad"] = identidad[descuadre]
    salida["diferencia_ingresos"] = diferencia[descuadre]
    return salida


def a_float(serie):
    # Los tipos nullable de pandas (Int64) rompen la asignación con máscara; se pasa a float64.
    return pd.Series(serie.to_numpy(dtype="float64", na_value=np.nan), index=serie.index, dtype="float64")


def cociente(numerador, denominador, indice):
    # Divide solo donde el denominador es > 0; en el resto queda nulo.
    resultado = pd.Series(np.nan, index=indice, dtype=float)
    validos = denominador > 0
    resultado[validos] = numerador[validos] / denominador[validos]
    return resultado


def calcular_indicadores_congestion(df):
    res = df.copy()
    ingresos = a_float(calcular_ingresos_totales(res))
    res["ingresos_totales"] = ingresos
    atendidas = a_float(res["atendidas"])
    resueltas = a_float(res["resueltas"])
    pend_fin = a_float(res["pendientes_fin"])
    nuevas = a_float(res["nuevas_ingresadas"])

    # Sin ingresos no hay tasa de resolución: queda nula, no 1.0.
    cr = cociente(resueltas, ingresos, res.index)
    res["tasa_resolucion"] = cr
    # Variante solo con nuevas_ingresadas, para comparar. No es el clearance rate.
    res["tasa_resolucion_solo_nuevas"] = cociente(resueltas, nuevas, res.index)
    res["tasa_congestion"] = cociente(atendidas, resueltas, res.index)

    tp = cociente(pend_fin, atendidas, res.index)
    tp[atendidas == 0] = 0.0
    res["tasa_pendencia"] = tp

    de = pd.Series(np.nan, index=res.index, dtype=float)
    validos = resueltas > 0
    de[validos] = (pend_fin[validos] / resueltas[validos]) * 365.0
    res["duracion_estimada_dias"] = de

    res["acumulacion_neta_anual"] = ingresos - resueltas

    res["flag_acumula_mora"] = (cr < 1.0).fillna(False)
    res["flag_sin_resolucion_anual"] = ((atendidas > 0) & (resueltas == 0)).fillna(False)
    res["flag_sin_movimiento"] = (atendidas == 0).fillna(False)

    if "personal_items_total" in res.columns:
        items = res["personal_items_total"].astype(float)
        res["atendidas_por_item_personal"] = np.where(items > 0, atendidas / items, np.nan)
    else:
        res["atendidas_por_item_personal"] = np.nan

    if "recurso_juzgados_publicados" in res.columns:
        juzgados = res["recurso_juzgados_publicados"].astype(float)
        res["atendidas_por_juzgado_territorio"] = np.where(juzgados > 0, atendidas / juzgados, np.nan)
    else:
        res["atendidas_por_juzgado_territorio"] = np.nan

    return res